# NexaMem Quick Start: File Storage

This notebook demonstrates how to use NexaMem with file-based storage for simple, portable AI memory management.

## 📁 Why File Storage?

File storage is perfect for:
- **Development & Testing**: Quick setup without external dependencies
- **Small Applications**: Lightweight memory management for prototypes
- **Portable Data**: Easy to backup, share, and version control
- **Local Development**: No need for database servers or Redis instances

## What You'll Learn

- Setting up NexaMem with file storage
- Basic memory operations (store, retrieve, search, delete)
- Working with memory scopes and organization
- File storage configuration options
- Performance considerations and best practices
- Data persistence and backup strategies

## 1. Environment Setup and Imports

Let's start by importing the necessary modules and setting up our environment.

In [ ]:
import sys
import os
from pathlib import Path
import tempfile
import shutil
import time
import json
from datetime import datetime
from typing import Dict, List, Optional, Any

# Add the src directory to Python path if running from examples directory
if Path.cwd().name == 'examples':
    project_root = Path.cwd().parent
    src_path = project_root / 'src'
    if str(src_path) not in sys.path:
        sys.path.insert(0, str(src_path))
        
print(f"✅ Python path updated. Working directory: {Path.cwd()}")
print(f"📂 Project structure ready for file storage demo")

In [ ]:
# Import NexaMem components for file storage
from nexamem.new_aimemory import AIMemory
from nexamem.config.aimemory import AIMemoryConfig
from nexamem.config.storage import StorageConfig, FileConfig
from nexamem.memory_scope import MemoryScope

print("✅ NexaMem modules imported successfully")
print("📁 File storage components ready")

## 2. File Storage Configuration

Let's configure NexaMem to use file-based storage with various options.

In [ ]:
# Create a temporary directory for our demo (you'd use a permanent path in real applications)
demo_storage_path = Path(tempfile.mkdtemp()) / "nexamem_file_demo"
demo_storage_path.mkdir(parents=True, exist_ok=True)

print(f"📁 Demo storage directory: {demo_storage_path}")

# Configure file storage with custom settings
file_config = FileConfig(
    path=str(demo_storage_path / "memories.json"),
    encoding="utf-8",
    backup_count=3,  # Keep 3 backup files
    auto_create_dirs=True  # Automatically create parent directories
)

# Create the storage configuration
storage_config = StorageConfig(
    type="file",
    config=file_config
)

# Create the main AI Memory configuration
config = AIMemoryConfig(
    storage=storage_config,
    channels_yaml=None,  # We'll register channels programmatically
    strict_yaml_validation=False
)

print("✅ File storage configuration created")
print(f"📄 Storage file: {file_config.path}")
print(f"🔢 Backup count: {file_config.backup_count}")
print(f"🔤 Encoding: {file_config.encoding}")

## 3. Initialize Memory System

Now let's initialize our file-based memory system.

In [ ]:
# Initialize the memory system with file storage
memory = AIMemory(config)

print("✅ File-based memory system initialized")

# Verify the storage file exists (it should be created automatically)
storage_file = Path(file_config.path)
if storage_file.exists():
    print(f"📄 Storage file created: {storage_file}")
    print(f"📊 Current file size: {storage_file.stat().st_size} bytes")
else:
    print(f"📄 Storage file will be created on first write: {storage_file}")

## 4. Basic Memory Operations

Let's start with basic memory operations using file storage.

In [ ]:
# Create a memory scope for our demo
scope = MemoryScope(
    env="development",
    agent_id="file_storage_demo",
    user_id="demo_user",
    session_id="file_session_001"
)

scope_string = f"{scope.env}:{scope.agent_id}:{scope.user_id}:{scope.session_id}"

print("✅ Memory scope created")
print(f"🔍 Scope string: {scope_string}")

# Store some sample memories
sample_memories = [
    {
        "content": "User wants to learn about file storage in NexaMem",
        "metadata": {"type": "user_query", "topic": "storage", "timestamp": datetime.now().isoformat()}
    },
    {
        "content": "File storage is ideal for development and small applications",
        "metadata": {"type": "system_response", "topic": "storage", "timestamp": datetime.now().isoformat()}
    },
    {
        "content": "User prefers simple setup without external dependencies",
        "metadata": {"type": "user_preference", "category": "technical", "timestamp": datetime.now().isoformat()}
    }
]

# Store memories and collect their IDs
memory_ids = []
for mem in sample_memories:
    memory_id = memory.store(
        content=mem["content"],
        scope=scope_string,
        metadata=mem["metadata"]
    )
    memory_ids.append(memory_id)
    print(f"📝 Stored memory: {memory_id[:8]}... - {mem['content'][:50]}...")

print(f"\n✅ Stored {len(memory_ids)} memories in file storage")

In [ ]:
# Check the storage file after writing
storage_file = Path(file_config.path)
if storage_file.exists():
    file_size = storage_file.stat().st_size
    print(f"📄 Storage file updated")
    print(f"📊 File size: {file_size} bytes ({file_size/1024:.2f} KB)")
    
    # Peek at the file structure (first 500 characters)
    try:
        with open(storage_file, 'r', encoding='utf-8') as f:
            content_preview = f.read(500)
    except PermissionError:
        print("⚠️ Cannot read file - it may be locked by another process")
        content_preview = "File content unavailable due to permission restrictions"
    except Exception as e:
        print(f"⚠️ Error reading file: {str(e)}")
        content_preview = f"Error reading file: {str(e)}"
    
    print(f"\n📋 File content preview:")
    print("=" * 50)
    print(content_preview[:300] + "..." if len(content_preview) > 300 else content_preview)
    print("=" * 50)
else:
    print("❌ Storage file not found")

## 5. Retrieving Memories

Let's explore different ways to retrieve memories from file storage.

In [ ]:
# Retrieve all memories for our scope
all_memory_ids = memory.list(scope=scope_string)

print(f"📋 Found {len(all_memory_ids)} memories in scope")
print("🔍 Retrieving all memories:")
print("=" * 60)

for i, memory_id in enumerate(all_memory_ids, 1):
    memory_data = memory.retrieve(memory_id)
    
    if memory_data:
        content = memory_data['content']
        metadata = memory_data.get('metadata', {})
        mem_type = metadata.get('type', 'unknown')
        
        print(f"{i}. [{mem_type}] {content}")
        print(f"   📍 ID: {memory_id[:12]}...")
        print(f"   📊 Metadata: {len(metadata)} fields")
        print()
    else:
        print(f"{i}. ❌ Memory not found: {memory_id}")

In [ ]:
# Retrieve a specific memory by ID
if memory_ids:
    specific_id = memory_ids[0]
    specific_memory = memory.retrieve(specific_id)
    
    print(f"🎯 Retrieving specific memory: {specific_id[:12]}...")
    
    if specific_memory:
        print("✅ Memory found:")
        print(f"📝 Content: {specific_memory['content']}")
        print(f"📊 Metadata keys: {list(specific_memory.get('metadata', {}).keys())}")
        print(f"🏷️ Type: {specific_memory.get('metadata', {}).get('type', 'N/A')}")
        print(f"📅 Timestamp: {specific_memory.get('metadata', {}).get('timestamp', 'N/A')}")
    else:
        print("❌ Memory not found")
else:
    print("❌ No memories to retrieve")

## 6. Searching Memories

File storage supports content-based searching. Let's explore search capabilities.

In [ ]:
# Search for memories containing specific terms
search_terms = ["storage", "user", "simple", "file"]

print("🔍 Testing search functionality:")
print("=" * 50)

for term in search_terms:
    print(f"\n🔎 Searching for '{term}':")
    
    try:
        search_results = memory.search(query=term, scope=scope_string)
        
        if search_results:
            print(f"   ✅ Found {len(search_results)} result(s)")
            
            # Show details for each result
            for result_id in search_results:
                result_memory = memory.retrieve(result_id)
                if result_memory:
                    content_preview = result_memory['content'][:60] + ('...' if len(result_memory['content']) > 60 else '')
                    mem_type = result_memory.get('metadata', {}).get('type', 'unknown')
                    print(f"      📄 [{mem_type}] {content_preview}")
                    print(f"         ID: {result_id[:12]}...")
        else:
            print(f"   🚫 No results found for '{term}'")
    
    except Exception as e:
        print(f"   ⚠️ Search error: {str(e)}")

print(f"\n✅ Search testing completed")

## 7. Performance Testing

Let's test the performance characteristics of file storage.

In [ ]:
# Performance test: Batch operations
print("⚡ File Storage Performance Test")
print("=" * 50)

# Generate test data
test_memories = []
for i in range(50):  # Reasonable size for file storage demo
    test_memories.append({
        "content": f"Test memory #{i+1}: This is sample content for performance testing with file storage backend.",
        "metadata": {
            "test_id": i+1,
            "batch": "performance_test",
            "size": "medium",
            "timestamp": datetime.now().isoformat()
        }
    })

# Test storage performance
print(f"📝 Testing storage of {len(test_memories)} memories...")
start_time = time.time()

test_memory_ids = []
for test_mem in test_memories:
    memory_id = memory.store(
        content=test_mem["content"],
        scope=scope_string,
        metadata=test_mem["metadata"]
    )
    test_memory_ids.append(memory_id)

storage_time = time.time() - start_time
print(f"✅ Storage completed in {storage_time:.3f} seconds")
print(f"📊 Average: {(storage_time/len(test_memories)*1000):.2f} ms per memory")

# Test retrieval performance
print(f"\n🔍 Testing retrieval of {len(test_memory_ids)} memories...")
start_time = time.time()

retrieved_count = 0
for memory_id in test_memory_ids:
    memory_data = memory.retrieve(memory_id)
    if memory_data:
        retrieved_count += 1

retrieval_time = time.time() - start_time
print(f"✅ Retrieved {retrieved_count} memories in {retrieval_time:.3f} seconds")
print(f"📊 Average: {(retrieval_time/retrieved_count*1000):.2f} ms per retrieval")

# Check final file size
storage_file = Path(file_config.path)
if storage_file.exists():
    final_size = storage_file.stat().st_size
    print(f"\n📄 Final storage file size: {final_size} bytes ({final_size/1024:.2f} KB)")
    print(f"📊 Average bytes per memory: {final_size/(len(memory_ids) + len(test_memory_ids)):.1f}")

In [ ]:
# Test search performance
print("\n🔍 Testing search performance...")

search_queries = ["test", "memory", "performance", "sample", "content"]
start_time = time.time()

total_results = 0
for query in search_queries:
    search_results = memory.search(query=query, scope=scope_string)
    total_results += len(search_results)

search_time = time.time() - start_time
print(f"✅ Executed {len(search_queries)} searches in {search_time:.3f} seconds")
print(f"📊 Average: {(search_time/len(search_queries)*1000):.2f} ms per search")
print(f"📋 Total results across all searches: {total_results}")

# Clean up test memories
print(f"\n🧹 Cleaning up {len(test_memory_ids)} test memories...")
start_time = time.time()

deleted_count = 0
for memory_id in test_memory_ids:
    if memory.delete(memory_id):
        deleted_count += 1

cleanup_time = time.time() - start_time
print(f"✅ Deleted {deleted_count} memories in {cleanup_time:.3f} seconds")
print(f"📊 Average: {(cleanup_time/deleted_count*1000):.2f} ms per deletion")

## 8. Data Persistence and File Management

Let's explore how file storage handles data persistence and file management.

In [ ]:
# Examine file structure and persistence
print("💾 File Storage Persistence & Management")
print("=" * 50)

storage_file = Path(file_config.path)
storage_dir = storage_file.parent

print(f"📁 Storage directory: {storage_dir}")
print(f"📄 Main storage file: {storage_file.name}")

# List all files in storage directory
print(f"\n📋 Files in storage directory:")
for file_path in sorted(storage_dir.iterdir()):
    if file_path.is_file():
        size = file_path.stat().st_size
        modified = datetime.fromtimestamp(file_path.stat().st_mtime).strftime('%Y-%m-%d %H:%M:%S')
        print(f"   📄 {file_path.name}: {size} bytes (modified: {modified})")

# Check current memory count
current_memories = memory.list(scope=scope_string)
print(f"\n📊 Current state:")
print(f"   🧠 Memories in scope: {len(current_memories)}")

# Demonstrate data persistence by creating a new memory instance
print(f"\n🔄 Testing data persistence...")
print("   Creating new memory instance with same configuration...")

# Create a new memory instance with the same configuration
memory2 = AIMemory(config)

# Check if we can still access the same memories
persistent_memories = memory2.list(scope=scope_string)
print(f"   ✅ New instance found {len(persistent_memories)} memories")

if len(persistent_memories) == len(current_memories):
    print("   🎯 Perfect persistence - all memories accessible!")
else:
    print(f"   ⚠️ Persistence issue - expected {len(current_memories)}, found {len(persistent_memories)}")

# Verify content of a specific memory
if persistent_memories:
    test_memory = memory2.retrieve(persistent_memories[0])
    if test_memory:
        print(f"   📝 Sample memory content: {test_memory['content'][:50]}...")
        print("   ✅ Content verification successful!")

## 9. File Storage Configuration Options

Let's explore different file storage configuration options and their effects.

In [ ]:
# Test different file storage configurations
print("⚙️ File Storage Configuration Options:")
print("=" * 50)

# Show current configuration
print(f"📁 Current storage path: {file_config.path}")
print(f"🔤 Encoding: {file_config.encoding}")
print(f"🔢 Backup count: {file_config.backup_count}")
print(f"📂 Auto-create directories: {file_config.auto_create_dirs}")

# Check for backup files
storage_dir = Path(file_config.path).parent
backup_files = list(storage_dir.glob("*.bak*"))

print(f"\n💾 Backup Management:")
if backup_files:
    print(f"   📋 Found {len(backup_files)} backup files:")
    for backup in sorted(backup_files):
        print(f"      📄 {backup.name} ({backup.stat().st_size} bytes)")
else:
    print(f"   📋 No backup files found (created on modifications)")

# File system information
storage_file = Path(file_config.path)
if storage_file.exists():
    stat = storage_file.stat()
    print(f"\n📊 File Information:")
    print(f"   📄 Size: {stat.st_size} bytes")
    print(f"   📅 Created: {datetime.fromtimestamp(stat.st_ctime).strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"   📝 Modified: {datetime.fromtimestamp(stat.st_mtime).strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"   🔐 Permissions: {oct(stat.st_mode)[-3:]}")

## 10. Backup and Recovery

File storage supports automatic backups. Let's explore backup functionality.

In [ ]:
# Demonstrate backup functionality
print("💾 File Storage Backup & Recovery")
print("=" * 50)

# Store some additional memories to trigger backups
backup_test_memories = [
    "Important data that should be backed up",
    "Critical user preference for backup testing", 
    "System configuration that needs protection"
]

print(f"📝 Adding {len(backup_test_memories)} memories to trigger backup creation...")

for i, content in enumerate(backup_test_memories):
    memory_id = memory.store(
        content=content,
        scope=scope_string,
        metadata={
            "backup_test": True,
            "sequence": i+1,
            "timestamp": datetime.now().isoformat()
        }
    )
    print(f"   ✅ Stored backup test memory {i+1}: {memory_id[:8]}...")

# Check for backup files after modifications
storage_dir = Path(file_config.path).parent
backup_files = list(storage_dir.glob("*.bak*"))

print(f"\n🔍 Backup files check:")
if backup_files:
    print(f"   📋 Found {len(backup_files)} backup files:")
    for backup in sorted(backup_files):
        size = backup.stat().st_size
        modified = datetime.fromtimestamp(backup.stat().st_mtime).strftime('%Y-%m-%d %H:%M:%S')
        print(f"      📄 {backup.name}: {size} bytes (created: {modified})")
else:
    print("   📋 No backup files found")
    print("   💡 Backups are created automatically when the main file is modified")

# Show backup configuration
print(f"\n⚙️ Backup Configuration:")
print(f"   🔢 Max backup files: {file_config.backup_count}")
print(f"   🔄 Backup rotation: Automatic (oldest files removed when limit exceeded)")
print(f"   📁 Backup location: Same directory as main storage file")

## 11. Storage Backend Comparison

Let's compare file storage with other storage options.

In [ ]:
# Storage comparison matrix
print("📊 Storage Backend Comparison")
print("=" * 60)

comparison_data = {
    "Feature": [
        "Setup Complexity",
        "External Dependencies", 
        "Data Persistence",
        "Concurrent Access",
        "Search Performance",
        "ACID Transactions",
        "Scalability",
        "Backup & Recovery",
        "Development Speed",
        "Production Ready"
    ],
    "File Storage": [
        "Very Simple",
        "None",
        "Excellent",
        "Limited",
        "Linear O(n)",
        "No",
        "Small-Medium",
        "Built-in",
        "Very Fast",
        "Small Apps"
    ],
    "SQLite": [
        "Simple",
        "None (embedded)",
        "Excellent", 
        "Good (reads)",
        "Indexed O(log n)",
        "Yes",
        "Medium-Large",
        "Manual",
        "Fast",
        "Yes"
    ],
    "Redis/Azure": [
        "Complex",
        "Redis Server",
        "Excellent",
        "Excellent",
        "Very Fast",
        "Limited",
        "Very Large",
        "Built-in",
        "Medium",
        "Enterprise"
    ]
}

# Print comparison table
for i, feature in enumerate(comparison_data["Feature"]):
    print(f"{feature:<20} | File: {comparison_data['File Storage'][i]:<15} | SQLite: {comparison_data['SQLite'][i]:<15} | Redis: {comparison_data['Redis/Azure'][i]}")

print(f"\n🎯 **Choose File Storage When:**")
print(f"   ✅ Building prototypes or small applications")
print(f"   ✅ No external dependencies allowed")
print(f"   ✅ Simple deployment requirements")
print(f"   ✅ Data portability is important")
print(f"   ✅ Development speed is priority")

print(f"\n🎯 **Consider Alternatives When:**")
print(f"   ⚠️ Need concurrent access from multiple processes")
print(f"   ⚠️ Require complex queries or indexing")
print(f"   ⚠️ Data volume exceeds 10K memories")
print(f"   ⚠️ Need ACID transaction guarantees")
print(f"   ⚠️ Building enterprise applications")

## 12. Best Practices and Summary

Let's review best practices and summarize what we've learned.

In [ ]:
# File storage best practices and final summary
print("📚 File Storage Best Practices & Summary")
print("=" * 50)

print("""
🎯 **Configuration Recommendations:**
   • Use absolute paths for storage files in production
   • Set backup_count to 3-5 for important data
   • Choose UTF-8 encoding for international character support
   • Enable auto_create_dirs for deployment flexibility

📊 **Performance Considerations:**
   • File storage is fastest for small to medium datasets (<10K memories)
   • Search performance degrades linearly with memory count
   • Consider SQLite for >10K memories or complex queries
   • Batch operations when possible for better performance

💾 **Data Management:**
   • Regular backups are created automatically
   • Monitor file size growth in production
   • Consider file rotation for very large datasets
   • Use meaningful file names and directory structures

🔐 **Security & Reliability:**
   • Set appropriate file permissions in production
   • Consider encrypting the storage file for sensitive data
   • Use version control for development environments
   • Implement monitoring for file size and access patterns

🚀 **Production Deployment:**
   • Use persistent storage (not temp directories)
   • Monitor disk space usage
   • Implement log rotation
   • Consider read-only replicas for scaling reads
""")

# Current demo statistics
current_memories = memory.list(scope=scope_string)
storage_file = Path(file_config.path)

print(f"📊 **Current Demo Statistics:**")
if storage_file.exists():
    file_size = storage_file.stat().st_size
    print(f"   📄 Storage file size: {file_size} bytes ({file_size/1024:.2f} KB)")
    print(f"   🧠 Total memories: {len(current_memories)}")
    print(f"   📈 Average memory size: {file_size/len(current_memories):.1f} bytes" if current_memories else "   📈 No memories to analyze")
    
    # Show directory contents
    storage_dir = storage_file.parent
    all_files = list(storage_dir.iterdir())
    print(f"   📁 Files created: {len([f for f in all_files if f.is_file()])}")
else:
    print("   ❌ Storage file not found")

print(f"   🔧 Configuration: {file_config.backup_count} backups, {file_config.encoding} encoding")

print(f"\n✅ **File Storage Demo Complete!**")

print(f"""
🎉 **What You've Learned:**

✅ **Setup & Configuration:**
   • How to configure NexaMem with file storage
   • File storage options and their effects
   • Directory and path management

✅ **Core Operations:**
   • Storing memories with metadata
   • Retrieving individual and multiple memories
   • Searching through stored content
   • Deleting unwanted memories

✅ **Advanced Features:**
   • Automatic backup creation and rotation
   • Data persistence across application restarts
   • Performance characteristics and optimization
   • File management and monitoring

✅ **Best Practices:**
   • When to choose file storage vs alternatives
   • Production deployment considerations
   • Security and reliability recommendations
   • Performance optimization strategies

🚀 **Next Steps:**
   • Try the SQLite storage notebook for more advanced features
   • Experiment with different file configurations
   • Build a small application using file storage
   • Consider scaling to SQLite or Redis for larger projects
""")

print(f"\n📁 Demo storage location: {demo_storage_path}")
print(f"💡 Storage files preserved for your examination")